In [7]:
"""
Timing estimate for the hierarchical sampler run.

Prints:
  - median single likelihood evaluation time
  - estimated total wall time for the emcee run
"""

import time
import numpy as np
from scipy.stats import gaussian_kde
from converse_likelihood import EOSHyperparameterLikelihood
from pprint import pprint as pp

# ---------- config (must match hierarchical.ipynb) ----------
N_WALKERS  = 64
N_STEPS    = 5000
N_TIMING   = 500   # likelihood calls used to estimate median time

log10_p1_cgs = 35.293288373856534
log10_p2_cgs = 35.648689969687915
causal       = False
mmin, mmax   = 1.0, 2.0

# injection truth — a point we know is valid
THETA_TRUE = {
    "param1": 4.183994058757201,
    "param2": 3.168076721819637,
    "param3": 1.300271383168368,
}

In [8]:

# ------------------------------------------------------------

data  = np.load("50_runs_test.npz")
covs  = data["covs"].astype(np.float64)
covs  = np.array([(c + c.T) / 2 + 1e-10 * np.eye(2) for c in covs])
means = data["means"]

viable = np.load("viable_gamma_samples.npy")
kde    = gaussian_kde(viable.T, bw_method="scott")

likelihood = EOSHyperparameterLikelihood(
    cov_matrices=covs,
    means=means,
    mass_min=mmin,
    mass_max=mmax,
    log10_pressure1_cgs=log10_p1_cgs,
    log10_pressure2_cgs=log10_p2_cgs,
    causal=causal,
)

# --- sanity check at truth ---
likelihood.parameters = THETA_TRUE.copy()
log_l_true = likelihood.log_likelihood()
print(f"log L at injection truth: {log_l_true:.4f}")
if not np.isfinite(log_l_true):
    print("WARNING: log L is not finite at the truth point — check pressure units.")

# --- sample N_TIMING points from the KDE (in-bounds, no viability filter needed) ---
lo = np.array([2.7, 1.0, 1.0])
hi = np.array([5.0, 5.0, 2.31])
sample_points = []
while len(sample_points) < N_TIMING:
    batch = kde.resample(N_TIMING * 5).T
    mask  = np.all((batch >= lo) & (batch <= hi), axis=1)
    sample_points.extend(batch[mask].tolist())
sample_points = sample_points[:N_TIMING]

log L at injection truth: 94.9833


In [9]:
# --- time each call individually ---
times = []
n_inf = 0
for p in sample_points:
    params = {"param1": p[0], "param2": p[1], "param3": p[2]}
    likelihood.parameters = params
    t0 = time.perf_counter()
    ll = likelihood.log_likelihood()
    times.append(time.perf_counter() - t0)
    if not np.isfinite(ll):
        n_inf += 1

times = np.array(times)
t_med  = np.median(times)
t_mean = np.mean(times)
t_p95  = np.percentile(times, 95)

print(f"\n--- single likelihood timing over {N_TIMING} KDE samples ---")
print(f"  median : {t_med*1e3:.2f} ms")
print(f"  mean   : {t_mean*1e3:.2f} ms")
print(f"  95th % : {t_p95*1e3:.2f} ms")
print(f"  -inf   : {n_inf}/{N_TIMING} ({100*n_inf/N_TIMING:.1f}% rejected EOS)")



--- single likelihood timing over 500 KDE samples ---
  median : 25.56 ms
  mean   : 25.46 ms
  95th % : 28.14 ms
  -inf   : 15/500 (3.0% rejected EOS)


In [5]:
# --- emcee cost model ---
# Each step evaluates the likelihood once per walker (proposals are independent).
# Total calls = n_walkers * n_steps
total_calls    = N_WALKERS * N_STEPS
t_total_median = total_calls * t_med
t_total_p95    = total_calls * t_p95

def fmt_time(seconds):
    if seconds < 60:
        return f"{seconds:.1f} s"
    elif seconds < 3600:
        return f"{seconds/60:.1f} min"
    else:
        return f"{seconds/3600:.2f} hr"

print(f"\n--- emcee estimate ({N_WALKERS} walkers × {N_STEPS} steps = {total_calls:,} calls) ---")
print(f"  using median time : {fmt_time(t_total_median)}")
print(f"  using 95th-pct    : {fmt_time(t_total_p95)}")
print()
print("Note: emcee is embarrassingly parallel across walkers per step.")
print(f"With n_cores workers the wall time scales as ~1/n_cores.")
for n_cores in [1, 2, 4, 8, 10]:
    print(f"  {n_cores:2d} core(s): {fmt_time(t_total_median / n_cores)}")



--- emcee estimate (64 walkers × 5000 steps = 320,000 calls) ---
  using median time : 2.38 hr
  using 95th-pct    : 3.29 hr

Note: emcee is embarrassingly parallel across walkers per step.
With n_cores workers the wall time scales as ~1/n_cores.
   1 core(s): 2.38 hr
   2 core(s): 1.19 hr
   4 core(s): 35.7 min
   8 core(s): 17.8 min
  10 core(s): 14.3 min


In [6]:
import cProfile
import pstats
import io

likelihood.parameters = THETA_TRUE.copy()

pr = cProfile.Profile()
pr.enable()
for _ in range(10):
    likelihood.log_likelihood()
pr.disable()

stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats("cumulative")
ps.print_stats(20)
print(stream.getvalue())

         7197 function calls (7191 primitive calls) in 0.451 seconds

   Ordered by: cumulative time
   List reduced from 200 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000    0.315    0.158 /Users/ved/miniforge3/envs/neutron/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3663(run_code)
      3/2    0.000    0.000    0.315    0.158 {built-in method builtins.exec}
        1    0.137    0.137    0.315    0.315 /var/folders/t6/9g0339bj7fjdlm1vw31kx_tc0000gn/T/ipykernel_82086/288430426.py:1(<module>)
       10    0.002    0.000    0.194    0.019 /Users/ved/Desktop/cambridge/research/neutron/pre-merger/pipeline/converse_likelihood.py:141(log_likelihood)
      500    0.001    0.000    0.162    0.000 /Users/ved/miniforge3/envs/neutron/lib/python3.12/site-packages/bilby/gw/conversion.py:783(lambda_from_mass_and_family)
        4    0.000    0.000    0.127    0.032 /Users/ved/miniforge3/envs/neutr